# Building & Deploying a GCP Agent with ADK
### A hands-on course notebook — from your first tool to a live agent you can chat with in the Google Cloud Console

**What we build:** a single agent equipped with tools to create/delete Cloud Storage buckets, upload/delete files,
create/delete BigQuery datasets and tables — plus Sessions, State, Streaming, Memory, RAG, and (optionally) the
BigQuery MCP Toolbox. We finish by deploying it to **Vertex AI Agent Engine**, where you and your students can
open the console and chat with it directly.

**Course map**

| # | Module | Concept |
|---|--------|---------|
| 0 | Setup | Auth, enable APIs |
| 1 | GCS bucket tools | Tools, Agent, Session, State |
| 2 | GCS file tools | Reading/writing State from inside a tool |
| 3 | BigQuery tools | Dataset & table create/delete |
| 4 | Streaming | Token-by-token responses |
| 5 | Memory | Long-term recall across sessions (Memory Bank) |
| 6 | RAG | Vertex AI RAG Engine as a retrieval tool |
| 7 | BigQuery MCP (optional) | Swapping hand-written tools for the managed MCP Toolbox |
| 8 | Assemble + Deploy | One root agent, deployed to Agent Engine |
| 9 | Console interaction + IAM + cleanup | Chat with it in the browser, then tear it down |

> **Note on naming:** Google announced at Cloud Next 2026 that Vertex AI's agent stack is being rebranded as the
> **Gemini Enterprise Agent Platform**. Under the hood the ADK Python framework and the **Agent Engine** runtime
> are the same thing this notebook uses — if console labels look slightly different from this write-up by the
> time you run it, that's why. Always cross-check against `https://adk.dev` if something doesn't match.


---
## Module 0 — Environment Setup

Before an agent can touch GCP, Colab needs to (a) authenticate as *you*, and (b) have the right APIs turned on
in the project. Skipping this is the #1 reason students see `PermissionDenied` errors.


In [ ]:
!pip install -q google-adk google-cloud-storage google-cloud-bigquery google-genai google-cloud-aiplatform

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os

PROJECT_ID = "himanshugcpproject"   # <-- change to your GCP project
LOCATION   = "us-central1"          # region for Vertex AI / Agent Engine

os.environ["GOOGLE_CLOUD_PROJECT"]      = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"]     = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"   # route Gemini calls through Vertex AI, not a personal API key

print(f"Authenticated. Project set to: {PROJECT_ID}")

In [ ]:
!gcloud services enable \
  storage.googleapis.com \
  bigquery.googleapis.com \
  aiplatform.googleapis.com \
  --project=$PROJECT_ID

print("APIs enabled.")

**Console check:** open `console.cloud.google.com` → **APIs & Services → Enabled APIs & services** and
confirm *Cloud Storage API*, *BigQuery API*, and *Vertex AI API* are listed.


---
## Module 1 — GCS Bucket Tools + Agent + Session + State

**Concept:** an ADK agent has three moving parts —
- **Tools**: plain Python functions. ADK reads the type hints + docstring to build the schema the LLM uses to call it.
- **Agent** (`LlmAgent`): wires a model + tools + instructions together.
- **Session / State / Runner**: the agent has no memory of its own. A `Session` is one conversation thread;
  `State` is a dict attached to it where the agent (or a tool) can stash facts. The `Runner` actually drives
  a turn through the event loop.


In [ ]:
from google.cloud import storage

def create_bucket(bucket_name: str, location: str = "US") -> dict:
    """Creates a new Google Cloud Storage bucket.

    Args:
        bucket_name: Globally unique name for the new bucket.
        location: GCS region/multi-region for the bucket (default "US").

    Returns:
        dict with status and bucket name, or an error message.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        bucket = client.create_bucket(bucket_name, location=location)
        return {"status": "success", "bucket": bucket.name, "location": location}
    except Exception as e:
        return {"status": "error", "message": str(e)}


def delete_bucket(bucket_name: str) -> dict:
    """Deletes a Google Cloud Storage bucket. The bucket must be empty.

    Args:
        bucket_name: Name of the bucket to delete.

    Returns:
        dict with status, or an error message.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        bucket = client.bucket(bucket_name)
        bucket.delete()
        return {"status": "success", "message": f"Bucket {bucket_name} deleted."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

> **Teaching point:** the docstring isn't decoration — ADK parses it to tell the LLM what the tool does and
> what each argument means. A vague docstring is the #1 cause of the model calling a tool incorrectly.


In [ ]:
from google.adk.agents import LlmAgent

MODEL = "gemini-2.5-flash"

storage_agent = LlmAgent(
    name="gcs_admin_agent",
    model=MODEL,
    description="An agent that manages Google Cloud Storage buckets.",
    instruction=(
        "You are a helpful GCP storage administrator. "
        "Use the tools available to create or delete buckets when asked. "
        "Always confirm the bucket name back to the user, and report tool "
        "errors clearly instead of guessing what happened."
    ),
    tools=[create_bucket, delete_bucket],
)

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()

APP_NAME   = "gcp_training_app"
USER_ID    = "student_01"
SESSION_ID = "session_01"

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state={"course": "GCP Agentic AI", "trainer": "Himanshu"},  # <-- initial STATE
)

runner = Runner(agent=storage_agent, app_name=APP_NAME, session_service=session_service)

print("Session created with initial state:", session.state)

In [ ]:
async def ask_agent(runner, user_id, session_id, message_text):
    """Sends one message to the agent and prints its final text response."""
    content = types.Content(role="user", parts=[types.Part(text=message_text)])
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.is_final_response():
            print("AGENT:", event.content.parts[0].text)

await ask_agent(runner, USER_ID, SESSION_ID,
    "Please create a storage bucket called himanshu-adk-demo-bucket-01 in the US region.")

**Console check:** Cloud Storage → Buckets — the new bucket should be there.

In [ ]:
await ask_agent(runner, USER_ID, SESSION_ID, "Actually, please delete that bucket now.")

**Console check:** refresh Cloud Storage → Buckets — it's gone.

---
## Module 2 — File Tools + Reading/Writing State

**Concept:** so far state has just sat there. Now we'll have a tool *write into it* using `ToolContext`, so the
agent remembers "the bucket we're working with" across turns without the student repeating the name.


In [ ]:
from google.adk.tools import ToolContext

def upload_file(bucket_name: str, destination_blob_name: str, file_contents: str,
                 tool_context: ToolContext) -> dict:
    """Uploads a small text file to a GCS bucket.

    Args:
        bucket_name: Name of the destination bucket.
        destination_blob_name: Desired filename (object name) in the bucket.
        file_contents: The text content to write into the file.
        tool_context: Injected automatically by ADK - gives access to session state.

    Returns:
        dict with status, or an error message.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(destination_blob_name)
        blob.upload_from_string(file_contents)

        # remember the last bucket/file we touched, so future turns don't need to repeat it
        tool_context.state["last_bucket"] = bucket_name
        tool_context.state["last_file"] = destination_blob_name

        return {"status": "success", "uploaded": destination_blob_name, "bucket": bucket_name}
    except Exception as e:
        return {"status": "error", "message": str(e)}


def delete_file(bucket_name: str, blob_name: str) -> dict:
    """Deletes a file (object) from a GCS bucket.

    Args:
        bucket_name: Name of the bucket containing the file.
        blob_name: Name of the file/object to delete.

    Returns:
        dict with status, or an error message.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        bucket = client.bucket(bucket_name)
        bucket.blob(blob_name).delete()
        return {"status": "success", "message": f"{blob_name} deleted from {bucket_name}."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

> **Teaching point:** notice `tool_context: ToolContext` — ADK injects this automatically when a tool
> declares that parameter. `tool_context.state[...]` is read/write, and any change is persisted back to
> the session after the tool runs. This is *the* mechanism for tools to leave breadcrumbs for later turns.


In [ ]:
storage_agent = LlmAgent(
    name="gcs_admin_agent",
    model=MODEL,
    description="An agent that manages Google Cloud Storage buckets and files.",
    instruction=(
        "You are a helpful GCP storage administrator. Use your tools to create/delete buckets "
        "and upload/delete files. If the user refers to 'that bucket' or 'that file', check "
        "session state for last_bucket / last_file before asking them to repeat themselves."
    ),
    tools=[create_bucket, delete_bucket, upload_file, delete_file],
)
runner = Runner(agent=storage_agent, app_name=APP_NAME, session_service=session_service)

In [ ]:
await ask_agent(runner, USER_ID, SESSION_ID,
    "Create a bucket called himanshu-adk-demo-bucket-02, then upload a file called notes.txt "
    "into it with the content 'GCP Agentic AI course notes'.")

# check what got written into state
updated_session = await session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
print("State now:", updated_session.state)

In [ ]:
# Notice we don't repeat the bucket/file name - a well-instructed agent will use state
await ask_agent(runner, USER_ID, SESSION_ID, "Now delete that file, then delete the bucket too.")

**Console check:** Cloud Storage → confirm the bucket was created with the file, then confirm both are gone after the second turn.

---
## Module 3 — BigQuery Tools

Same pattern, new API. We add dataset and table create/delete as tools, and merge them into one combined agent
so students see a single assistant managing two different GCP services.


In [ ]:
from google.cloud import bigquery
from typing import List, Dict

def create_dataset(dataset_id: str, location: str = "US") -> dict:
    """Creates a new BigQuery dataset.

    Args:
        dataset_id: Name for the new dataset (no spaces).
        location: BigQuery dataset location (default "US").

    Returns:
        dict with status, or an error message.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        dataset = bigquery.Dataset(f"{PROJECT_ID}.{dataset_id}")
        dataset.location = location
        dataset = client.create_dataset(dataset, exists_ok=False)
        return {"status": "success", "dataset": dataset.dataset_id, "location": location}
    except Exception as e:
        return {"status": "error", "message": str(e)}


def delete_dataset(dataset_id: str, delete_contents: bool = True) -> dict:
    """Deletes a BigQuery dataset.

    Args:
        dataset_id: Name of the dataset to delete.
        delete_contents: If True, deletes tables inside the dataset too (default True).

    Returns:
        dict with status, or an error message.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        client.delete_dataset(f"{PROJECT_ID}.{dataset_id}",
                               delete_contents=delete_contents, not_found_ok=False)
        return {"status": "success", "message": f"Dataset {dataset_id} deleted."}
    except Exception as e:
        return {"status": "error", "message": str(e)}


def create_table(dataset_id: str, table_id: str, schema_fields: List[Dict[str, str]]) -> dict:
    """Creates a BigQuery table with a given schema.

    Args:
        dataset_id: Dataset that will contain the table.
        table_id: Name of the new table.
        schema_fields: List of columns, each like {"name": "id", "type": "STRING"}.
            Valid types: STRING, INTEGER, FLOAT, BOOLEAN, TIMESTAMP, DATE.

    Returns:
        dict with status, or an error message.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        schema = [bigquery.SchemaField(f["name"], f["type"]) for f in schema_fields]
        table = bigquery.Table(f"{PROJECT_ID}.{dataset_id}.{table_id}", schema=schema)
        table = client.create_table(table)
        return {"status": "success", "table": table.table_id, "columns": len(schema)}
    except Exception as e:
        return {"status": "error", "message": str(e)}


def delete_table(dataset_id: str, table_id: str) -> dict:
    """Deletes a BigQuery table.

    Args:
        dataset_id: Dataset containing the table.
        table_id: Name of the table to delete.

    Returns:
        dict with status, or an error message.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        client.delete_table(f"{PROJECT_ID}.{dataset_id}.{table_id}", not_found_ok=False)
        return {"status": "success", "message": f"Table {table_id} deleted."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

> **Teaching point:** `schema_fields: List[Dict[str, str]]` is deliberately simple JSON-friendly typing —
> ADK's function-tool schema generation supports `str`, `int`, `float`, `bool`, `list`, and `dict`, but nested
> typed objects (like a custom `SchemaField` class) don't translate cleanly. Keep tool signatures in plain
> JSON-shaped types and this class of bug disappears.


In [ ]:
gcp_agent = LlmAgent(
    name="gcp_admin_agent",
    model=MODEL,
    description="An agent that manages GCS buckets/files and BigQuery datasets/tables.",
    instruction=(
        "You are a GCP administrator assistant. Use your tools to manage Cloud Storage "
        "buckets/files and BigQuery datasets/tables. Confirm every action back to the user "
        "and surface tool errors verbatim rather than guessing what went wrong."
    ),
    tools=[create_bucket, delete_bucket, upload_file, delete_file,
           create_dataset, delete_dataset, create_table, delete_table],
)
runner = Runner(agent=gcp_agent, app_name=APP_NAME, session_service=session_service)

In [ ]:
await ask_agent(runner, USER_ID, SESSION_ID,
    "Create a BigQuery dataset called adk_demo_dataset in the US, then create a table "
    "called students with columns: id (INTEGER), name (STRING), score (FLOAT).")

**Console check:** BigQuery console → your project → `adk_demo_dataset` → `students` table with the three columns.

In [ ]:
await ask_agent(runner, USER_ID, SESSION_ID,
    "Delete the students table, then delete the whole adk_demo_dataset.")

**Console check:** refresh BigQuery console — dataset is gone.

---
## Module 4 — Streaming

So far `ask_agent` waits for the *final* response. ADK actually streams partial tokens as the model generates
them (`event.partial == True`) — this is what powers a "typing" effect in a chat UI. Let's watch it happen.


In [ ]:
async def ask_agent_streaming(runner, user_id, session_id, message_text):
    """Prints the agent's response as it streams, token chunk by token chunk."""
    content = types.Content(role="user", parts=[types.Part(text=message_text)])
    run_config = types.RunConfig() if hasattr(types, "RunConfig") else None

    print("AGENT: ", end="")
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "text", None):
                    print(part.text, end="", flush=True)
    print()  # newline once the stream ends

await ask_agent_streaming(runner, USER_ID, SESSION_ID,
    "In one short paragraph, explain what BigQuery is used for.")

> **Teaching point:** ADK's `Runner.run_async` always yields events incrementally — whether you treat that
> as "streaming" is really about whether you print each `event` as it arrives (this cell) or wait for
> `event.is_final_response()` (Module 1's `ask_agent`). For a production chat UI you'd stream these events
> straight to the frontend over a websocket/SSE connection instead of `print`.


---
## Module 5 — Memory (long-term recall across sessions)

**Concept — don't confuse this with State from Module 2.** State lives inside *one* session (one conversation
thread) and disappears when the session ends. **Memory** is the layer above it: facts an agent should recall
even in a *brand-new* session, days later — "the last time we spoke, we created a dataset called X."

ADK's `PreloadMemoryTool` automatically searches long-term memory and injects relevant facts into context before
each turn. The catch: **the fully managed backing store — Vertex AI Memory Bank — only activates once the agent
is deployed to Agent Engine** (Module 8 sets `GOOGLE_CLOUD_AGENT_ENGINE_ID`, which is what wires this up). For
now we add the tool so it's part of the agent's toolset; we'll see it actually persist after deployment.


In [ ]:
from google.adk.tools.preload_memory_tool import PreloadMemoryTool

gcp_agent = LlmAgent(
    name="gcp_admin_agent",
    model=MODEL,
    description="An agent that manages GCS and BigQuery resources, with long-term memory.",
    instruction=(
        "You are a GCP administrator assistant with memory of past conversations. "
        "Use your tools to manage Cloud Storage and BigQuery resources. Check memory for "
        "context from earlier sessions (e.g. resources you created previously) before asking "
        "the user to repeat information."
    ),
    tools=[create_bucket, delete_bucket, upload_file, delete_file,
           create_dataset, delete_dataset, create_table, delete_table,
           PreloadMemoryTool()],
)
print("Agent now has", len(gcp_agent.tools), "tools, including long-term memory.")

---
## Module 6 — RAG (Retrieval-Augmented Generation)

**Concept:** give the agent a searchable knowledge base — e.g. your own GCP training notes — so it can answer
questions grounded in *your* material instead of general Gemini knowledge. We'll use **Vertex AI RAG Engine**:
upload a couple of text snippets into a RAG corpus, then hand the agent a retrieval tool pointed at that corpus.


In [ ]:
from vertexai import rag
import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)

# 1. Create a RAG corpus
rag_corpus = rag.create_corpus(display_name="gcp_training_notes")
print("Created corpus:", rag_corpus.name)

In [ ]:
# 2. Add a couple of sample training documents (swap these for your real course notes / PDFs)
import tempfile, os as _os

sample_docs = {
    "bigquery_basics.txt": (
        "BigQuery is Google Cloud's serverless, highly scalable data warehouse. "
        "It separates storage and compute, supports standard SQL, and can query "
        "petabyte-scale datasets without managing infrastructure."
    ),
    "gcs_basics.txt": (
        "Cloud Storage is Google Cloud's object storage service. Buckets are containers "
        "for objects (files), each bucket has a globally unique name, and storage classes "
        "(Standard, Nearline, Coldline, Archive) trade off cost against access frequency."
    ),
}

tmp_paths = []
for filename, content in sample_docs.items():
    path = _os.path.join(tempfile.gettempdir(), filename)
    with open(path, "w") as f:
        f.write(content)
    tmp_paths.append(path)

for path in tmp_paths:
    rag.upload_file(corpus_name=rag_corpus.name, path=path)

print("Uploaded", len(tmp_paths), "documents to the RAG corpus.")

In [ ]:
# 3. Give the agent a retrieval tool pointed at this corpus
from google.adk.tools.retrieval.vertex_ai_rag_retrieval import VertexAiRagRetrieval

course_notes_retrieval = VertexAiRagRetrieval(
    name="retrieve_course_notes",
    description="Searches the GCP training course notes (BigQuery, Cloud Storage fundamentals).",
    rag_resources=[rag.RagResource(rag_corpus=rag_corpus.name)],
    similarity_top_k=3,
)

gcp_agent = LlmAgent(
    name="gcp_admin_agent",
    model=MODEL,
    description="A GCP administrator + course-notes assistant.",
    instruction=(
        "You are a GCP administrator assistant. Use your admin tools to manage GCS/BigQuery "
        "resources. Use retrieve_course_notes to answer conceptual questions about GCP services "
        "grounded in the course material before relying on general knowledge."
    ),
    tools=[create_bucket, delete_bucket, upload_file, delete_file,
           create_dataset, delete_dataset, create_table, delete_table,
           PreloadMemoryTool(), course_notes_retrieval],
)
runner = Runner(agent=gcp_agent, app_name=APP_NAME, session_service=session_service)

await ask_agent(runner, USER_ID, SESSION_ID,
    "According to the course notes, what's the difference between BigQuery and Cloud Storage?")

**Console check:** Vertex AI → RAG Engine → your corpus `gcp_training_notes` should list the two uploaded files.

---
## Module 7 — BigQuery MCP Toolbox *(optional / advanced)*

Everything in Module 3 was **hand-written** tools — you own the code, the auth, and the maintenance. Google
also publishes the **MCP Toolbox for Databases**: a standalone server that exposes a pre-built, versioned
BigQuery toolset over the Model Context Protocol. Instead of writing `create_dataset` yourself, your agent
connects to this server and gets a maintained toolset for free.

This needs a **separate running process** (the toolbox binary or a Cloud Run service) — not something that
lives inside a Colab cell — so treat this as a "here's how you'd wire it in" reference rather than something
to run live in class today.


In [ ]:
# Reference only - run this OUTSIDE Colab, e.g. in Cloud Shell or a terminal:
#
#   curl -O https://storage.googleapis.com/genai-toolbox/latest/linux/amd64/toolbox
#   chmod +x toolbox
#   BIGQUERY_PROJECT=himanshugcpproject ./toolbox --prebuilt bigquery --stdio
#
# Then, back in your ADK agent code, connect to it as an MCP toolset:

from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

bigquery_mcp_toolset = MCPToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="./toolbox",
            args=["--prebuilt", "bigquery", "--stdio"],
            env={"BIGQUERY_PROJECT": PROJECT_ID},
        )
    )
)

# You would then simply add `bigquery_mcp_toolset` to an agent's tools=[...] list
# alongside (or instead of) the hand-written create_dataset/create_table tools from Module 3.
print("Reference snippet only - requires the toolbox binary running as a separate process.")

> **Discussion point for class:** ask students *why* you'd choose MCP over hand-written tools. Answer:
> centralized maintenance (Google patches the toolset, not you), reuse across multiple agents, and a single
> place to enforce access policy — at the cost of an extra process to run and operate.


---
## Module 8 — Assemble the Final Agent and Deploy to Agent Engine

Now we bring every tool from Modules 1–6 into one `root_agent`, write it out as a proper ADK **agent package**
(a folder with `__init__.py`, `agent.py`, `requirements.txt`), and deploy it with a single `adk deploy` command.
Once deployed, Agent Engine gives you a hosted, autoscaled endpoint **and a console UI you can chat with**.

**Why a folder, not just Colab cells?** `adk deploy agent_engine` packages a self-contained agent module and
ships it to the cloud — it needs real files on disk, not variables living in a notebook kernel.


In [ ]:
import os

AGENT_DIR = "/content/gcp_admin_agent"
os.makedirs(AGENT_DIR, exist_ok=True)

with open(f"{AGENT_DIR}/__init__.py", "w") as f:
    f.write("from . import agent\n")

print("Created", AGENT_DIR)

In [ ]:
agent_py_source = f'''"""Final GCP admin agent: GCS + BigQuery tools, memory, and RAG."""
import os
from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.cloud import storage, bigquery
from typing import List, Dict

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = "gemini-2.5-flash"


def create_bucket(bucket_name: str, location: str = "US") -> dict:
    """Creates a new Google Cloud Storage bucket.

    Args:
        bucket_name: Globally unique name for the new bucket.
        location: GCS region/multi-region for the bucket (default "US").
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        bucket = client.create_bucket(bucket_name, location=location)
        return {{"status": "success", "bucket": bucket.name, "location": location}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def delete_bucket(bucket_name: str) -> dict:
    """Deletes a Google Cloud Storage bucket. The bucket must be empty.

    Args:
        bucket_name: Name of the bucket to delete.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        client.bucket(bucket_name).delete()
        return {{"status": "success", "message": f"Bucket {{bucket_name}} deleted."}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def upload_file(bucket_name: str, destination_blob_name: str, file_contents: str,
                 tool_context: ToolContext) -> dict:
    """Uploads a small text file to a GCS bucket.

    Args:
        bucket_name: Name of the destination bucket.
        destination_blob_name: Desired filename (object name) in the bucket.
        file_contents: The text content to write into the file.
        tool_context: Injected automatically by ADK.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        bucket = client.bucket(bucket_name)
        bucket.blob(destination_blob_name).upload_from_string(file_contents)
        tool_context.state["last_bucket"] = bucket_name
        tool_context.state["last_file"] = destination_blob_name
        return {{"status": "success", "uploaded": destination_blob_name, "bucket": bucket_name}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def delete_file(bucket_name: str, blob_name: str) -> dict:
    """Deletes a file (object) from a GCS bucket.

    Args:
        bucket_name: Name of the bucket containing the file.
        blob_name: Name of the file/object to delete.
    """
    client = storage.Client(project=PROJECT_ID)
    try:
        client.bucket(bucket_name).blob(blob_name).delete()
        return {{"status": "success", "message": f"{{blob_name}} deleted from {{bucket_name}}."}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def create_dataset(dataset_id: str, location: str = "US") -> dict:
    """Creates a new BigQuery dataset.

    Args:
        dataset_id: Name for the new dataset (no spaces).
        location: BigQuery dataset location (default "US").
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        dataset = bigquery.Dataset(f"{{PROJECT_ID}}.{{dataset_id}}")
        dataset.location = location
        dataset = client.create_dataset(dataset, exists_ok=False)
        return {{"status": "success", "dataset": dataset.dataset_id, "location": location}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def delete_dataset(dataset_id: str, delete_contents: bool = True) -> dict:
    """Deletes a BigQuery dataset.

    Args:
        dataset_id: Name of the dataset to delete.
        delete_contents: If True, deletes tables inside the dataset too.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        client.delete_dataset(f"{{PROJECT_ID}}.{{dataset_id}}",
                               delete_contents=delete_contents, not_found_ok=False)
        return {{"status": "success", "message": f"Dataset {{dataset_id}} deleted."}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def create_table(dataset_id: str, table_id: str, schema_fields: List[Dict[str, str]]) -> dict:
    """Creates a BigQuery table with a given schema.

    Args:
        dataset_id: Dataset that will contain the table.
        table_id: Name of the new table.
        schema_fields: List of columns like {{"name": "id", "type": "STRING"}}.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        schema = [bigquery.SchemaField(f["name"], f["type"]) for f in schema_fields]
        table = bigquery.Table(f"{{PROJECT_ID}}.{{dataset_id}}.{{table_id}}", schema=schema)
        table = client.create_table(table)
        return {{"status": "success", "table": table.table_id, "columns": len(schema)}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


def delete_table(dataset_id: str, table_id: str) -> dict:
    """Deletes a BigQuery table.

    Args:
        dataset_id: Dataset containing the table.
        table_id: Name of the table to delete.
    """
    client = bigquery.Client(project=PROJECT_ID)
    try:
        client.delete_table(f"{{PROJECT_ID}}.{{dataset_id}}.{{table_id}}", not_found_ok=False)
        return {{"status": "success", "message": f"Table {{table_id}} deleted."}}
    except Exception as e:
        return {{"status": "error", "message": str(e)}}


root_agent = LlmAgent(
    name="gcp_admin_agent",
    model=MODEL,
    description="An agent that manages GCS buckets/files and BigQuery datasets/tables, with memory.",
    instruction=(
        "You are a GCP administrator assistant. Use your tools to create/delete GCS buckets and "
        "files, and create/delete BigQuery datasets and tables. Confirm every action back to the "
        "user, use session state to avoid asking the user to repeat bucket/file names you already "
        "know, and check memory for context from earlier conversations. Report tool errors verbatim."
    ),
    tools=[create_bucket, delete_bucket, upload_file, delete_file,
           create_dataset, delete_dataset, create_table, delete_table,
           PreloadMemoryTool()],
)
'''

with open(f"{AGENT_DIR}/agent.py", "w") as f:
    f.write(agent_py_source)

with open(f"{AGENT_DIR}/requirements.txt", "w") as f:
    f.write("google-adk\ngoogle-cloud-storage\ngoogle-cloud-bigquery\n")

print("Wrote agent.py, __init__.py, requirements.txt to", AGENT_DIR)
!ls -la $AGENT_DIR

> We left the RAG retrieval tool (Module 6) and the MCP toolbox (Module 7) out of this deployed version to
> keep the first deployment simple — once this works, add `course_notes_retrieval` / `bigquery_mcp_toolset`
> back into `root_agent`'s `tools=[...]` list in `agent.py` and redeploy with the same command below.


In [ ]:
# Agent Engine stages deployment artifacts in a GCS bucket - create one if you don't have one
STAGING_BUCKET_NAME = f"{PROJECT_ID}-agent-engine-staging"
!gsutil mb -l $LOCATION -p $PROJECT_ID gs://$STAGING_BUCKET_NAME 2>/dev/null || echo "Bucket may already exist, continuing."

In [ ]:
!adk deploy agent_engine \
  --project=$PROJECT_ID \
  --region=$LOCATION \
  --staging_bucket=gs://$STAGING_BUCKET_NAME \
  --display_name="GCP Admin Training Agent" \
  --trace_to_cloud \
  $AGENT_DIR

**Deployment takes 5-10 minutes.** When it finishes, the CLI prints something like:

```
AgentEngine created. Resource name: projects/123456789/locations/us-central1/reasoningEngines/7516195...
```

Copy that resource name — you'll use it in Module 9 both for IAM and for reconnecting to the agent from a
future notebook session.


---
## Module 9 — Grant Permissions, Chat via the Console, Observe, Clean Up

### 9a. IAM — the deployed agent runs as a *different* identity than you

Locally, tools ran using **your** authenticated Colab identity. Once deployed, the agent runs as Agent Engine's
service agent, which does **not** automatically have Storage/BigQuery permissions. Grant them explicitly:


In [ ]:
PROJECT_NUMBER = !gcloud projects describe $PROJECT_ID --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]
AGENT_ENGINE_SA = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"

!gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member="serviceAccount:{AGENT_ENGINE_SA}" \
  --role="roles/storage.admin"

!gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member="serviceAccount:{AGENT_ENGINE_SA}" \
  --role="roles/bigquery.admin"

print("Granted Storage Admin and BigQuery Admin to", AGENT_ENGINE_SA)

> **Teaching point:** in a real course/production setting, `roles/storage.admin` and `roles/bigquery.admin`
> are broad — call this out to students and mention narrower custom roles (e.g. just
> `storage.buckets.create/delete`, `bigquery.datasets.create/delete`) as the "do this properly" follow-up.

### 9b. Interact via the Google Cloud Console (no code)

1. Open **console.cloud.google.com** → search for **Agent Engine** (or **Vertex AI → Agent Engine**).
2. You'll see **"GCP Admin Training Agent"** in the list — click it.
3. Open the **Test Agent** / **Playground** panel — this is a chat box right in the browser.
4. Try: *"Create a bucket called console-test-bucket-01"* and watch it execute for real.

This is the moment to tell students: *this is the same agent, same tools, same code — just running as a managed
cloud service instead of inside your Colab kernel.*

### 9c. Interact programmatically (to prove the console isn't doing anything special)


In [ ]:
import vertexai
from vertexai import agent_engines

vertexai.init(project=PROJECT_ID, location=LOCATION)

# paste the resource name printed by `adk deploy` above
AGENT_ENGINE_RESOURCE_NAME = "projects/PROJECT_NUMBER/locations/us-central1/reasoningEngines/AGENT_ENGINE_ID"

remote_agent = agent_engines.get(AGENT_ENGINE_RESOURCE_NAME)

for event in remote_agent.stream_query(
    user_id="student_01",
    message="List the tools you have available and what each one does.",
):
    print(event)

### 9d. Observability

Because we deployed with `--trace_to_cloud`, every tool call is automatically traced.

**Console check:** **Cloud Trace → Trace Explorer** — find a recent trace, open it, and you'll see each tool
invocation as a span with latency. Great for showing students *why* an agent felt slow, not just *that* it was.

### 9e. Cleanup — avoid ongoing charges

Run these once the class/demo is done:


In [ ]:
# Delete the deployed agent
remote_agent.delete()

# Delete the RAG corpus from Module 6
rag.delete_corpus(name=rag_corpus.name)

# Delete the staging bucket
!gsutil -m rm -r gs://$STAGING_BUCKET_NAME

print("Cleanup complete.")

---
## Recap

You now have a single agent that:
- Creates/deletes GCS buckets and files, and BigQuery datasets/tables — via **custom function tools**
- Carries context within a conversation via **Session/State**
- Streams partial output like a real chat UI
- Recalls facts across separate conversations via **Memory Bank**
- Answers grounded questions from your own docs via **RAG**
- Can optionally swap in Google's managed **BigQuery MCP Toolbox** instead of hand-written tools
- Is **deployed to Vertex AI Agent Engine**, reachable from the Cloud Console, the Python SDK, or a REST API,
  with **Cloud Trace** observability and proper **IAM** for its service identity

That's the full loop from "agent as a notebook experiment" to "agent as a governed cloud service" — the same
journey students will need for any real ADK project.
